In [ ]:
import pandas as pd
import numpy as np

# 1. Load Master Data
maintorder_df = pd.read_csv(
    "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/SHAREPOINT/tbl_maintenance_order.csv",
    low_memory=False
)

# 2. FIX: Include "ID" in your column selection before renaming
maintorder_df = maintorder_df[["ID", "maintenanceorder_no", "maintenance_type", "operation_workcenter", "date_closed"]]
maintorder_df = maintorder_df[maintorder_df["maintenance_type"] == "YCM1"]
maintorder_df = maintorder_df.rename(columns={"ID": "MO_ID"})

# 3. Load Response Data
response_cm_df = pd.read_csv("D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/SHAREPOINT/tbl_response_cm.csv")
response_cm_df = response_cm_df.rename(columns={"ID": "CM_ID"})

# 4. FIX: Use a "right" join to ensure all response records are kept
merged_df = pd.merge(maintorder_df, response_cm_df, on="maintenanceorder_no", how="right")

# View the results
merged_df.drop_duplicates(inplace=True)
merged_df
# response_cm_df

In [ ]:
missing_workcenters = merged_df[(merged_df["maintenanceorder_no"].isnull())]
# missing_workcenters = missing_workcenters[["CM_ID", "maintenanceorder_no", "operation_workcenter"]]
missing_workcenters.to_csv("D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/missing_workcenters.csv", index=False, sep="|")

In [ ]:
unmapped_merged_df = merged_df[merged_df["maintenanceorder_no"].isnull()]
unmapped_merged_df

In [7]:
import pandas as pd
import numpy as np

cm_paths = [
        # "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/sharepoint-migration/system_hierarchy/RSD_CM_Data_batch1.csv",
        # "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/sharepoint-migration/system_hierarchy/RSD_CM_Data_batch2.csv",
        # "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/sharepoint-migration/system_hierarchy/RSD_CM_Data_batch3.csv",
        "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/sharepoint-migration/system_hierarchy/PSD_CM_Data.csv",
        "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/sharepoint-migration/system_hierarchy/SNC_CM_Data.csv",
        "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/sharepoint-migration/system_hierarchy/TNM_CM_Data.csv",
    ]

# combine all those files into a single dataframe,
cm_data = pd.concat([pd.read_csv(path, sep='|') for path in cm_paths], ignore_index=True)
cm_data.drop(columns=['check_system', 'check_sub_system', 'check_sub_sub_system', 'check_root_cause'], inplace=True)
cm_data = cm_data.rename(columns={"work_order_no": "maintenanceorder_no"})
cm_data

,maintenanceorder_no,notification_no,created_on,date_closed,station,system,sub_system,sub_sub_system,root_cause,remarks
0,4000448518,11901607,2022-01-02 02:13:16,2022-01-02 02:13:16,NaN,TPSS,AARU,Display Panel,Power supply failure,Incident Description: AARU trippedFailure: nan
1,4000445168,11896478,2022-01-15 06:04:29,2022-01-15 06:04:29,BFZ,Stinger,Front control panel,Others,Others,"Incident Description: OCC : 0500hrs,OCC report..."
2,4000445166,11896848,2022-01-16 05:55:39,2022-01-16 05:55:39,HAH,UPS,40 KVA (Station),Board,"Capasitor burn, Power surge","Incident Description: OCC : 0546hrs, SO fuad a..."
3,4000452440,11912924,2022-02-27 05:12:57,2022-02-27 05:12:57,HAH,TPSS,2000 kW Rectifier,Control System,"Power supply failure, equipment faulty","Incident Description: OCC; 0453hrs, CE Javeed ..."
4,4000452439,11913141,2022-02-28 08:42:29,2022-02-28 08:42:29,MKU,UPS,40 KVA (Station),Cable,"Insulation is cracked or brittle, Cable fault","Incident Description: OCC: 0841hrs, SO Suriyan..."
...,...,...,...,...,...,...,...,...,...,...
1644,4000573141,12213162,2023-12-18 10:24:04,2023-12-18 10:24:04,DEPOT,SWITCH 5 & 6,Switchdeck & RC Beam,RC Beam,NaN,NaN
1645,4000573604,12213960,2023-12-20 10:59:14,2023-12-20 10:59:14,DEPOT,GUIDEWAY BEAM,Earthing plate,Earthing bracket,NaN,NaN
1646,4000574222,12215790,2023-12-25 11:34:05,2023-12-25 11:34:05,MNL,GUIDEWAY BEAM STRUCTURE,Expended Metal,CHECKER PLATE LENGTH 123.5CM x WIDE 245.8CM,NaN,NaN
1647,4000574964,12217592,2023-12-28 15:55:54,2023-12-28 15:55:54,MNL,GUIDEWAY BEAM,Beam,Expansion joint plate,NaN,NaN


In [ ]:
# 1. Rename cm_data columns to match merged_df's naming
cm_data_temp = cm_data.rename(columns={
    'train_number': 'train_no',
    'station': 'station_name'
})

# 2. Columns to match on (AND logic)
match_cols = [
    'train_no', 'car_position', 'station_name',
    'system', 'sub_system', 'sub_sub_system', 'root_cause'
]

# 3. Fill NaNs with a dummy string so groupby/merge treats them as equal, not as mismatches
cm_data_temp[match_cols] = cm_data_temp[match_cols].fillna('MATCH_NAN')
unmapped_merged_df = unmapped_merged_df.copy()
unmapped_merged_df[match_cols] = unmapped_merged_df[match_cols].fillna('MATCH_NAN')

# 4. Group cm_data by the match columns, aggregate maintenanceorder_no as comma-separated candidates
aggregated_cm = cm_data_temp.groupby(match_cols)['maintenanceorder_no'].apply(
    lambda x: ', '.join(x.dropna().astype(str).str.replace(r'\.0$', '', regex=True).unique())
).reset_index()

aggregated_cm = aggregated_cm.rename(columns={'maintenanceorder_no': 'possible_maintenanceorder_no'})

# 5. Merge candidates into unmapped_merged_df
unmapped_merged_df = unmapped_merged_df.merge(aggregated_cm, on=match_cols, how='left')

# 6. Revert dummy strings back to true NaN
unmapped_merged_df[match_cols] = unmapped_merged_df[match_cols].replace('MATCH_NAN', np.nan)

unmapped_merged_df

In [ ]:
for col in match_cols:
    overlap = set(cm_data_temp[col].astype(str).str.strip().str.lower()) & set(unmapped_merged_df[col].astype(str).str.strip().str.lower())
    print(col, "-> overlapping values:", len(overlap))

In [ ]:
for col in ['system', 'sub_system', 'sub_sub_system', 'station_name']:
    print(f"--- cm_data_temp['{col}'] ---")
    print(cm_data_temp[col].dropna().unique()[:10])
    print()

for col in ['system', 'sub_system', 'sub_sub_system', 'station_name']:
    print(f"--- unmapped_merged_df['{col}'] ---")
    print(unmapped_merged_df[col].dropna().unique()[:10])
    print()